# Phase 2-4: Feature Engineering & Data Merging
# المرحلة 2-4: هندسة السمات ودمج البيانات

This notebook documents how we:
1. Extracted allele frequencies from gnomAD
2. Extracted 22+ pre-computed features from dbNSFP
3. Merged all three sources into one dataset


## Data Flow
## تدفق البيانات
ClinVar (2.5M variants, labels)
+ gnomAD (allele frequencies)
+ dbNSFP (conservation, physicochemical features)
─── INNER JOIN on variant_key ───→ 283K missense variants


In [ ]:
# Load all intermediate files
import pandas as pd

clinvar = pd.read_parquet('../data/intermediate/clinvar_labeled_clean.parquet')
gnomad = pd.read_parquet('../data/intermediate/gnomad_af_clean.parquet')
dbnsfp = pd.read_parquet('../data/intermediate/dbnsfp_selected_features.parquet')

print('=' * 60)
print('SOURCE DATASETS')
print('=' * 60)
print(f'ClinVar:  {len(clinvar):>12,} variants')
print(f'gnomAD:   {len(gnomad):>12,} variants')
print(f'dbNSFP:   {len(dbnsfp):>12,} variants')


## gnomAD: Allele Frequency Features
## سمات تردد الطفرة من نوماد

gnomAD provides population allele frequencies from 140,000+ individuals.
If a variant is common (AF > 1%), it is likely benign.

Derived features:
| Feature | Description |
|---------|-------------|
| AF | Global allele frequency |
| AF_popmax | Highest AF across populations |
| log_AF | log10(AF + 1e-8) for better distribution |
| is_common | Boolean: AF > 0.01 |


In [ ]:
# Show gnomAD feature summary
print('gnomAD Feature Summary:')
print(f'  Total variants:     {len(gnomad):,}')
print(f"  Common (AF > 0.01): {gnomad['is_common'].sum():,}")
print(f"  Rare (AF <= 0.01):  {(~gnomad['is_common']).sum():,}")
print(f"  Mean AF:            {gnomad['AF'].mean():.6f}")
print(f"  Median AF:          {gnomad['AF'].median():.6f}")


## dbNSFP: Pre-computed Features
## السمات المحسوبة مسبقا من دي بي ان اس اف بي

dbNSFP provides pre-computed features for every possible missense variant.
We extracted features in 3 categories:

### Conservation Scores (how conserved is this position across species?)
- phyloP100way, phyloP30way, phastCons100way, phastCons30way
- GERP++_RS, GERP++_NR

### Physicochemical Properties (how different is the amino acid change?)
- Grantham distance, BLOSUM62 score
- Polarity/Volume/Charge changes

### Amino Acid Properties
- Hydrophobicity, Molecular Weight, pI (for both ref and alt amino acids)

**CRITICAL:** We excluded REVEL, ClinPred, MetaSVM, and similar scores
because they are trained on ClinVar labels (circularity = cheating).


In [ ]:
# Show dbNSFP features extracted
feature_categories = {
    'Conservation': ['phyloP100way_vertebrate', 'phyloP30way_mammalian',
                     'phastCons100way_vertebrate', 'phastCons30way_mammalian',
                     'GERP++_RS', 'GERP++_NR'],
    'Physicochemical': ['Grantham_distance', 'BLOSUM62_score',
                        'polarity_change', 'volume_change', 'charge_change'],
    'Amino Acid': ['hydrophobicity_ref', 'hydrophobicity_alt',
                   'molecular_weight_ref', 'molecular_weight_alt',
                   'pI_ref', 'pI_alt', 'volume_ref', 'volume_alt',
                   'polarity_ref', 'polarity_alt', 'charge_ref', 'charge_alt']
}

print('=' * 60)
print('dbNSFP FEATURES EXTRACTED')
print('=' * 60)
total = 0
for category, features in feature_categories.items():
    existing = [f for f in features if f in dbnsfp.columns]
    total += len(existing)
    print(f'\n{category} ({len(existing)} features):')
    for f in existing:
        missing_pct = dbnsfp[f].isna().mean() * 100 if f in dbnsfp.columns else 100
        print(f'  {f:<35} (missing: {missing_pct:.1f}%)')

print(f'\nTotal features: {total}')


## Merge Strategy
## استراتيجية الدمج

1. **INNER JOIN** ClinVar with dbNSFP → keeps only missense variants
   (dbNSFP only contains missense by design, so this acts as our missense filter)
2. **LEFT JOIN** with gnomAD → adds AF features (AF=0 for unmatched = rare variant)


In [ ]:
# Show merge results
merged = pd.read_parquet('../data/processed/merged_clinvar_gnomad_dbnsfp.parquet')

print('=' * 60)
print('MERGE RESULTS')
print('=' * 60)
print(f'ClinVar rows:              {len(clinvar):>12,}')
print(f'After dbNSFP INNER JOIN:   {len(merged):>12,}  (missense only)')
print(f'Rows filtered out:         {len(clinvar)-len(merged):>12,}  (non-missense)')
print()
print(f"gnomAD matched:            {(merged['AF'] > 0).sum():>12,}")
print(f"gnomAD unmatched (AF=0):   {(merged['AF'] == 0).sum():>12,}")
print()
print(f'Final dataset: {len(merged):,} rows x {len(merged.columns)} columns')
print()
print('Class distribution after merge:')
print(f"  Pathogenic (1): {(merged['label']==1).sum():>10,} ({(merged['label']==1).mean()*100:.1f}%)")
print(f"  Benign (0):     {(merged['label']==0).sum():>10,} ({(merged['label']==0).mean()*100:.1f}%)")


In [ ]:
# Visualization: Missing values heatmap
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
feature_cols = [c for c in merged.columns if c not in
                ['chr','pos','ref','alt','gene','label','variant_key',
                 'ClinicalSignificance','PhenotypeIDS','review_stars',
                 'MolecularConsequence','has_dbnsfp_features']]
missing = merged[feature_cols].isnull().mean().sort_values(ascending=True) * 100
missing.plot(kind='barh', ax=ax, color='#3498db')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values per Feature')
ax.axvline(x=20, color='red', linestyle='--', label='20% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
